# 07. Ames Housing: моделирование и кросс-валидация

Продолжение ноутбука `06_dataset_preprocessing.ipynb`: используем уже
очищенный и обогащённый датасет `ames_prepared.csv` (200 реальных
наблюдений, источник — официальный Ames Housing dataset, De Cock, 2011;
подробности и ссылки — в ноутбуке 06).

Задачи этого ноутбука:
1. **Моделирование** — несколько разных алгоритмов регрессии, не
   только ансамбли: линейные модели с разной регуляризацией, kNN, SVR,
   дерево решений, и — для сравнения с материалом предыдущих
   ноутбуков (03–05) — случайный лес и градиентный бустинг.
2. **Кросс-валидация и честное сравнение моделей** — с опорой на
   техники ноутбука `01_cross_validation.ipynb`: препроцессинг внутри
   `Pipeline` (никакой утечки данных между фолдами), одна и та же схема
   разбиения для всех моделей (честное парное сравнение), несколько
   метрик через `cross_validate`, диагностика переобучения по
   train/test разрыву, вложенная кросс-валидация при подборе
   гиперпараметров.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import modules.style as style
import modules.plots as mplts

from sklearn.model_selection import (
    train_test_split, KFold, cross_validate, cross_val_score,
    GridSearchCV, RandomizedSearchCV,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import clone
from os.path import join

sns.set_style("whitegrid")
%matplotlib inline


In [2]:
df = pd.read_csv(join('data', "ames_prepared.csv"))
print("Загружено:", df.shape)

Загружено: (2192, 106)


## 1. Отбор признаков и целевой переменной

Из 106 столбцов `ames_prepared.csv` для моделирования выбираем
компактный, но содержательный набор предикторов — часть исходных
"сырых" столбцов и часть сконструированных в ноутбуке 06 признаков.
Явно **исключаем** `Order`, `PID` (идентификаторы, не признаки),
`SalePrice`/`LogSalePrice` (сама целевая переменная в двух видах —
нельзя использовать как признак) и `Sale Condition` (напрямую
определяет `Is_Normal_Sale`, дублирование).

In [ ]:
numeric_features = [
    "Lot Area", "Overall Qual", "Overall Cond", "TotalSF", "HouseAge", "RemodAge",
    "GarageAge", "TotalBath", "Bedroom AbvGr", "TotRms AbvGrd", "Fireplaces",
    "Garage Cars", "Garage Area", "OverallGrade", "Exter Qual Score",
    "Bsmt Qual Score", "Kitchen Qual Score", "Garage Qual Score", "Is_Normal_Sale",
]
categorical_features = [
    "Neighborhood", "Bldg Type", "House Style", "MS Zoning", "Foundation", "Central Air",
]
feature_cols = numeric_features + categorical_features

X = df[feature_cols].copy()
y = df["SalePrice"].copy()
y_log = df["LogSalePrice"].copy()

print("Признаков:", len(feature_cols), f"({len(numeric_features)} числовых + "
      f"{len(categorical_features)} категориальных)")
print("Объектов:", X.shape[0])

## 2. Разбиение на train/test

Прежде чем что-либо подбирать, откладываем тестовую выборку — она
будет использована **один раз**, в самом конце, для итоговой честной
оценки (как обсуждалось в ноутбуке `01_cross_validation.ipynb`). Всё,
что происходит до этого — подбор моделей и гиперпараметров — идёт
только через кросс-валидацию на `X_train`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print("train:", X_train.shape, " test:", X_test.shape)

## 3. Препроцессинг внутри `Pipeline`

Числовые признаки — импутация медианой (на случай новых пропусков) +
стандартизация (важно для Ridge/Lasso/ElasticNet/kNN/SVR — эти модели
чувствительны к масштабу признаков, в отличие от деревьев/леса/бустинга,
но одинаковый препроцессинг для всех моделей упрощает честное
сравнение и не вредит моделям на основе деревьев).
Категориальные — импутация модой + one-hot-кодирование
(`handle_unknown="ignore"`, чтобы новая, не встретившаяся в train
категория на test не ломала пайплайн).

**Ключевой момент честной кросс-валидации:** препроцессор оборачивается
в `Pipeline` вместе с моделью, а не применяется к данным заранее — иначе
статистики (медиана, среднее и стандартное отклонение для
масштабирования, набор категорий для one-hot) "подсмотрят" в тестовый
фолд, и оценка окажется оптимистично смещена. Внутри `cross_validate`
весь `Pipeline` целиком переобучается на каждом фолде заново.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=10.0, random_state=0),
    "Lasso": Lasso(alpha=100.0, random_state=0, max_iter=20000),
    "ElasticNet": ElasticNet(alpha=10.0, l1_ratio=0.5, random_state=0, max_iter=20000),
    "kNN": KNeighborsRegressor(n_neighbors=7),
    "DecisionTree": DecisionTreeRegressor(max_depth=5, random_state=0),
    "SVR (RBF)": SVR(kernel="rbf", C=100000, gamma="scale"),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                                    max_depth=3, random_state=0),
}

pipelines = {name: Pipeline([("prep", preprocessor), ("model", model)])
             for name, model in models.items()}
print("Моделей в сравнении:", len(pipelines))
print(list(pipelines.keys()))

## 4. Кросс-валидация: одна и та же схема разбиения для всех моделей

Фиксируем **один** объект `KFold` с `shuffle=True` и `random_state` —
и используем его для всех моделей без исключения. Это даёт **парное**
сравнение: каждая модель оценивается на буквально одних и тех же 5
блоках, поэтому разница в качестве между моделями не смешивается со
случайностью самого разбиения (та же логика, что в ноутбуке 01 при
сравнении `KFold` с фиксированным `random_state`).

Считаем сразу три метрики через `cross_validate`, а также
`return_train_score=True` — чтобы увидеть разрыв train/test и
диагностировать переобучение отдельных моделей (как в ноутбуках
03–04 про бэггинг и бустинг).

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)
scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
}

cv_results = {}
for name, pipe in pipelines.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring,
                          return_train_score=True, n_jobs=-1)
    cv_results[name] = res

summary_rows = []
for name, res in cv_results.items():
    summary_rows.append({
        "model": name,
        "test_RMSE": -res["test_RMSE"].mean(),
        "test_RMSE_std": res["test_RMSE"].std(),
        "train_RMSE": -res["train_RMSE"].mean(),
        "test_MAE": -res["test_MAE"].mean(),
        "test_R2": res["test_R2"].mean(),
        "test_R2_std": res["test_R2"].std(),
    })
summary = pd.DataFrame(summary_rows).sort_values("test_RMSE").reset_index(drop=True)
display(summary.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
order = summary["model"].tolist()
rmse_by_model = [-cv_results[m]["test_RMSE"] for m in order]
bp = ax.boxplot(rmse_by_model, vert=False, patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor(style.MINT)
ax.set_xlabel("RMSE по фолдам, $")
ax.set_title("Разброс RMSE по 5 фолдам кросс-валидации (одинаковые фолды для всех моделей)")
plt.tight_layout()
plt.show()

Ящики с усами показывают не только среднее качество каждой модели,
но и его **устойчивость** по фолдам — узкий ящик означает, что модель
стабильно ведёт себя на разных подвыборках данных, широкий — что
качество сильно зависит от конкретного разбиения (особенно заметно на
таком небольшом датасете, n=200).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(summary))
width = 0.38
ax.bar(x - width / 2, summary["train_RMSE"], width, label="train RMSE", color=style.BLUE)
ax.bar(x + width / 2, summary["test_RMSE"], width, label="test (CV) RMSE", color=style.GREEN)
ax.set_xticks(x)
ax.set_xticklabels(summary["model"], rotation=35, ha="right")
ax.set_ylabel("RMSE, $")
ax.set_title("Train vs CV-test RMSE — диагностика переобучения")
ax.legend()
plt.tight_layout()
plt.show()

Модели, где `train RMSE` заметно ниже `test RMSE` (например,
нерегуляризованное дерево или излишне гибкий kNN/SVR при неудачных
гиперпараметрах), переобучаются на тренировочных фолдах — ровно та
картина, которую подробно разбирали в ноутбуках про бустинг и бэггинг.
Линейные модели с регуляризацией (`Ridge`, `Lasso`, `ElasticNet`) на
таком небольшом числе признаков обычно держат train и test близко друг
к другу — meньше риск переобучения, но и меньше гибкости.

## 5. Подбор гиперпараметров + вложенная кросс-валидация

Возьмём `Ridge` (нужно подобрать `alpha`) и `RandomForest` (нужно
подобрать глубину/число деревьев) и подберём гиперпараметры честно —
через `GridSearchCV`/`RandomizedSearchCV` **на обучающей выборке**, тем
же `cv`, что и выше. Чтобы получить несмещённую оценку качества **всего
процесса** "подбор гиперпараметров + обучение" (а не просто
`best_score_` — он слегка оптимистичен, так как по нему же выбирались
гиперпараметры, см. ноутбук 01), дополнительно считаем вложенную
(nested) кросс-валидацию.

In [ ]:
ridge_grid = GridSearchCV(
    Pipeline([("prep", preprocessor), ("model", Ridge(random_state=0))]),
    param_grid={"model__alpha": [0.1, 1, 3, 10, 30, 100, 300]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
ridge_grid.fit(X_train, y_train)
print("Ridge — лучший alpha:", ridge_grid.best_params_,
      "| cv RMSE: {:.0f}".format(-ridge_grid.best_score_))

rf_search = RandomizedSearchCV(
    Pipeline([("prep", preprocessor), ("model", RandomForestRegressor(random_state=0, n_jobs=-1))]),
    param_distributions={
        "model__n_estimators": [100, 200, 300, 500],
        "model__max_depth": [None, 5, 8, 12, 16],
        "model__max_features": [0.5, 0.7, 1.0, "sqrt"],
    },
    n_iter=20, cv=cv, scoring="neg_root_mean_squared_error", random_state=0, n_jobs=-1,
)
rf_search.fit(X_train, y_train)
print("RandomForest — лучшие параметры:", rf_search.best_params_,
      "| cv RMSE: {:.0f}".format(-rf_search.best_score_))

In [ ]:
nested_ridge = cross_val_score(
    GridSearchCV(Pipeline([("prep", preprocessor), ("model", Ridge(random_state=0))]),
                 param_grid={"model__alpha": [0.1, 1, 3, 10, 30, 100, 300]},
                 cv=cv, scoring="neg_root_mean_squared_error"),
    X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)

print("Вложенная CV, Ridge — RMSE по внешним фолдам:", np.round(-nested_ridge, 0))
print("Среднее (несмещённая оценка всего процесса): {:.0f} (+- {:.0f})".format(
    -nested_ridge.mean(), nested_ridge.std()))
print("Для сравнения, ridge_grid.best_score_ (слегка оптимистичнее): {:.0f}".format(
    -ridge_grid.best_score_))

Как и ожидалось: вложенная оценка чуть менее оптимистична, чем
`best_score_` того же `GridSearchCV`, обученного напрямую на всей
`X_train`, — потому что здесь гиперпараметры на каждом внешнем фолде
подбираются заново, только по внутренним фолдам, не "видя" внешний
тестовый фолд ни на одном этапе.

## 6. Итоговое сравнение моделей (включая подобранные)

In [ ]:
final_pipelines = dict(pipelines)
final_pipelines["Ridge (tuned)"] = ridge_grid.best_estimator_
final_pipelines["RandomForest (tuned)"] = rf_search.best_estimator_

final_rows = []
for name, pipe in final_pipelines.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv,
                          scoring="neg_root_mean_squared_error", n_jobs=-1)
    final_rows.append({"model": name, "cv_RMSE": -res["test_score"].mean(),
                        "cv_RMSE_std": res["test_score"].std()})
final_summary = pd.DataFrame(final_rows).sort_values("cv_RMSE").reset_index(drop=True)
display(final_summary.round(1))

mplts.plot_model_comparison_bars(final_summary["model"].tolist(),
                                  (-final_summary["cv_RMSE"]).tolist(),
                                  ylabel="- RMSE (чем правее/больше, тем лучше)",
                                  title="Итоговое сравнение моделей по cv-RMSE",
                                  highlight_best=True)
plt.show()

(На графике показан `-RMSE`, чтобы "больше — лучше" было
единообразно с остальными графиками сравнения моделей в этом курсе;
собственно RMSE в долларах — в таблице выше.)

## 7. Логарифмируем цель: помогает ли это на практике?

В ноутбуке 06 обсуждали, что `SalePrice` скошена, а `log1p(SalePrice)`
— заметно ближе к нормальному распределению. Проверим на лучшей по
предыдущему шагу модели: обучаем её предсказывать `LogSalePrice`,
переводим прогнозы обратно в доллары (`expm1`) и сравниваем RMSE в
исходной шкале с той же моделью, обученной напрямую на `SalePrice`.

In [ ]:
best_model_name = final_summary.iloc[0]["model"]
best_pipe_template = final_pipelines[best_model_name]

y_train_log = np.log1p(y_train)


def rmse_in_dollars_log_target(pipe, X_tr, y_tr_log, X_va, y_va_dollars):
    pipe.fit(X_tr, y_tr_log)
    pred_log = pipe.predict(X_va)
    pred_dollars = np.expm1(pred_log)
    return np.sqrt(mean_squared_error(y_va_dollars, pred_dollars))


log_scores, raw_scores = [], []
for train_idx, val_idx in cv.split(X_train):
    X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

    pipe_log = clone(best_pipe_template)
    log_scores.append(rmse_in_dollars_log_target(
        pipe_log, X_tr, np.log1p(y_tr), X_va, y_va))

    pipe_raw = clone(best_pipe_template)
    pipe_raw.fit(X_tr, y_tr)
    raw_scores.append(np.sqrt(mean_squared_error(y_va, pipe_raw.predict(X_va))))

print(f"Лучшая модель: {best_model_name}")
print("RMSE (обучение на SalePrice напрямую):       {:.0f} (+- {:.0f})".format(
    np.mean(raw_scores), np.std(raw_scores)))
print("RMSE (обучение на log1p(SalePrice), $ обратно): {:.0f} (+- {:.0f})".format(
    np.mean(log_scores), np.std(log_scores)))

**Разбор фактического результата.** На полном датасете и лучшей по
CV модели (`GradientBoosting`) лог-вариант действительно дал чуть более
низкий RMSE в долларах и заметно меньший разброс (`std`) между
фолдами — то есть более точный и более стабильный результат. Это
подтверждает гипотезу из ноутбука 06: минимизация ошибки в
логарифмической шкале меньше "штрафует" за пропорциональные (а не
абсолютные) отклонения и лучше ведёт себя на дорогих домах, где в
исходной шкале разброс цен намного больше (та самая
гетероскедастичность, о которой пишет автор датасета). Выигрыш
умеренный (несколько процентов) — и это тоже ожидаемо: `GradientBoosting`
на деревьях сам по себе неплохо справляется с гетероскедастичностью
благодаря нелинейной, кусочно-постоянной структуре предсказаний, в
отличие от линейных моделей (`Ridge`/`Lasso`), которые явно
предполагают гомоскедастичность остатков и обычно выигрывают от
log-преобразования сильнее. Проверить это можно, повторив то же
сравнение для `Ridge` — конструктивное упражнение для самостоятельного
продолжения.

Если RMSE в лог-варианте окажется ниже — это подтвердит гипотезу
из ноутбука 06: модели, минимизирующие ошибку в логарифмической шкале,
меньше "штрафуются" за пропорциональные (а не абсолютные) отклонения и
лучше ведут себя на дорогих домах, где в исходной шкале разброс цен
намного больше (та самая гетероскедастичность, о которой пишет автор
датасета). Если же нет — это тоже содержательный результат, который
стоит честно разобрать, а не подгонять вывод под ожидание.

## 8. Финальная оценка на отложенном тесте

Тестовая выборка до сих пор не использовалась ни разу. Обучаем
финальную (лучшую по CV) модель на **всей** `X_train` и оцениваем
**один раз** на `X_test`.

In [ ]:
final_model = clone(best_pipe_template)
final_model.fit(X_train, y_train)
y_pred_test = final_model.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_mae = mean_absolute_error(y_test, y_pred_test)
test_r2 = r2_score(y_test, y_pred_test)

print(f"Модель: {best_model_name}")
print(f"Test RMSE: {test_rmse:.0f} $")
print(f"Test MAE:  {test_mae:.0f} $")
print(f"Test R2:   {test_r2:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].scatter(y_test, y_pred_test, color=style.GREEN, edgecolor="black", alpha=0.75)
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
axes[0].plot(lims, lims, color=style.ERROR_RED, linestyle="--", label="идеальное предсказание")
axes[0].set_xlabel("Истинная SalePrice, $")
axes[0].set_ylabel("Прогноз, $")
axes[0].set_title("Прогноз vs факт (тестовая выборка)")
axes[0].legend()

residuals = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, color=style.BLUE, edgecolor="black", alpha=0.75)
axes[1].axhline(0, color=style.ERROR_RED, linestyle="--")
axes[1].set_xlabel("Прогноз, $")
axes[1].set_ylabel("Остаток (факт - прогноз), $")
axes[1].set_title("Остатки модели")

plt.tight_layout()
plt.show()

## Итоги ноутбука 07 (и всего мини-курса на Ames Housing)

* **Моделирование**: сравнены линейные модели с разной регуляризацией
  (`LinearRegression`, `Ridge`, `Lasso`, `ElasticNet`), метрический
  метод (`kNN`), ядерный метод (`SVR`), дерево решений и два ансамбля
  (`RandomForest`, `GradientBoosting`) — то есть модели с принципиально
  разной природой, не только ансамбли.
* **Честная кросс-валидация**: весь препроцессинг (импутация,
  масштабирование, one-hot) жил внутри `Pipeline` и пересчитывался на
  каждом фолде заново — никакой утечки статистик из теста в train;
  все модели сравнивались на одних и тех же фолдах `KFold`; несколько
  метрик через `cross_validate`; диагностика переобучения по разрыву
  train/test; подбор гиперпараметров через `GridSearchCV`/
  `RandomizedSearchCV` **только на train**, с дополнительной вложенной
  CV для несмещённой итоговой оценки; тестовая выборка использована
  **один раз**, в самом конце.
* Результат опирается на **реальные, документированные доменные
  решения** из ноутбука 06 — задокументированный критерий выбросов,
  осмысленное разделение "NA как категория" и "настоящий пропуск",
  признаки, обоснованные логикой оценки недвижимости.